# 🚜 Proyecto Pastor Guardián — Detector Multi-Clase de Animales de Granja

## Descripción
Pipeline completo para entrenar un modelo **YOLO11** capaz de detectar **5 clases** en tiempo real:

| ID | Clase | Descripción |
|----|-------|-------------|
| 0 | `zorro` | 🦊 Depredador — **activa alerta** |
| 1 | `oveja` | 🐑 Oveja / guanaco / sheep |
| 2 | `gallina` | 🐔 Gallina / chicken (cualquier raza) |
| 3 | `cuy` | 🐹 Cuy / cobayo / guinea pig |
| 4 | `conejo` | 🐇 Conejo / rabbit / kelinci |

El modelo se entrena con **3 variantes de visión** (Normal, Infrarroja, Nocturna) para que funcione de día, de noche y en condiciones de baja luminosidad.

### Estructura
| Celda | Descripción |
|-------|-------------|
| **0** | Configuración global |
| **1** | Importaciones y hardware |
| **2** | Auditoría de datasets originales |
| **3** | Funciones de limpieza y conversión |
| **4** | Consolidación del dataset unificado (5 clases + 3 visiones) |
| **5** | Generación del `data.yaml` |
| **6** | Entrenamiento del modelo YOLO11 |
| **7** | Validación y métricas |
| **8** | Visualización de resultados |
| **9** | Exportación del modelo final |

> ⚠️ Ejecutar en orden. El modelo resultante se guarda en `output/unificado/`.

---
## Celda 0 — Configuración global

In [14]:
import os
from pathlib import Path
from datetime import datetime

BASE_DIR = Path(os.getcwd())
print(f"Directorio de trabajo: {BASE_DIR}")

# ─── Clases del modelo unificado ───
CLASS_NAMES = ["zorro", "oveja", "gallina", "cuy", "conejo"]
NUM_CLASSES = len(CLASS_NAMES)
ALERT_CLASS = 0  # zorro → activa alarma

# ─── Mapeo: animal -> nuevo ID de clase ───
CLASS_MAP = {
    "ZORROS": 0,
    "OVEJAS": 1,
    "GALLINAS": 2,
    "CUYES": 3,
    "CONEJOS": 4,
}

# ─── Visiones disponibles (todas se incluyen en entrenamiento) ───
VISIONS = ["NORMAL", "INFRARROJA", "NOCTURNO"]
SPLITS = ["train", "test", "valid"]

# ─── Directorios ───
UNIFIED_DIR = BASE_DIR / "dataset_unificado"
OUTPUT_DIR = BASE_DIR / "output"

# ─── Hiperparámetros ───
CONFIG = {
    "model_name": "yolo11n.pt",  # nano en vez de medium (5x más rápido)
    "batch_size": 8,            # el doble de batch
    "optimizer": "SGD",          # SGD es más rápido que AdamW
    "epochs": 100,               # menos épocas, early stopping igual corta antes
    "imgsz": 640,
    "patience": 25,
    "conf_thres": 0.25,
    "iou_thres": 0.45,
    "device": "cpu",
    "workers": 4,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3,
    "cos_lr": True,
    "close_mosaic": 10,
    "augment": True,
    "seed": 42,
    "project": str(OUTPUT_DIR / "runs"),
    "name": f"pastor_guardian_{datetime.now().strftime('%Y%m%d_%H%M')}",
}

print("\nConfiguración:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

Directorio de trabajo: d:\SOLEDAD VERSION FINAL

Configuración:
  model_name: yolo11n.pt
  batch_size: 8
  optimizer: AdamW
  epochs: 100
  imgsz: 640
  patience: 25
  conf_thres: 0.25
  iou_thres: 0.45
  device: cpu
  workers: 4
  lr0: 0.001
  lrf: 0.01
  momentum: 0.937
  weight_decay: 0.0005
  warmup_epochs: 3
  cos_lr: True
  close_mosaic: 10
  augment: True
  seed: 42
  project: d:\SOLEDAD VERSION FINAL\output\runs
  name: pastor_guardian_20260617_1102


---
## Celda 1 — Importaciones y verificación de hardware

In [2]:
import sys
import json
import shutil
import random
import warnings
import yaml
import numpy as np
import cv2
from collections import Counter, defaultdict
from pathlib import Path

warnings.filterwarnings("ignore")

try:
    import torch
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    from PIL import Image
    from ultralytics import YOLO
    from IPython.display import display, Video, Image as IPImage, clear_output
    import pandas as pd
    from tqdm.notebook import tqdm
    print("\U0001f7e2 Todas las dependencias instaladas.")
except ImportError as e:
    print(f"\u274c Falta: {e}")
    print("pip install ultralytics opencv-python matplotlib pillow pandas tqdm pyyaml")
    raise

# ─── Hardware ───
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
cuda = torch.cuda.is_available()
print(f"CUDA: {cuda}")
if cuda:
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    CONFIG["device"] = "cuda:0"
else:
    CONFIG["device"] = "cpu"
print(f"Dispositivo: {CONFIG['device']}")

🟢 Todas las dependencias instaladas.
Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.5.1+cu121
CUDA: True
  GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Dispositivo: cuda:0


---
## Celda 2 — Auditoría de los datasets consolidados

In [3]:
def audit_consolidated(animal, base_dir):
    info = {"animal": animal, "visiones": {}}
    for vision in VISIONS:
        info["visiones"][vision] = {}
        for split in SPLITS:
            img_dir = base_dir / animal / vision / split
            lbl_dir = img_dir / "labels"
            imgs = sum(1 for f in img_dir.glob("*") if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp"})
            lbls = sum(1 for f in lbl_dir.glob("*.txt") if f.stat().st_size > 0)
            info["visiones"][vision][split] = {"imagenes": imgs, "labels": lbls}
    return info

print("=" * 70)
print("AUDITORÍA DE DATASETS CONSOLIDADOS")
print("=" * 70)

total_global = {"train": 0, "test": 0, "valid": 0}
for animal, new_id in CLASS_MAP.items():
    info = audit_consolidated(animal, BASE_DIR)
    print(f"\n\U0001f4c1 {animal} -> clase {new_id} ({CLASS_NAMES[new_id]})")
    for vision, splits in info["visiones"].items():
        parts = []
        for split, data in splits.items():
            parts.append(f"{split}={data['imagenes']}img/{data['labels']}lbl")
            total_global[split] += data['imagenes']
        print(f"   [{vision}] " + " | ".join(parts))

print(f"\n{'=' * 70}")
print("TOTAL GLOBAL (3 visiones combinadas):")
for split, count in total_global.items():
    print(f"  {split}: {count} imágenes")
print(f"  Total general: {sum(total_global.values())} imágenes")

AUDITORÍA DE DATASETS CONSOLIDADOS

📁 ZORROS -> clase 0 (zorro)
   [NORMAL] train=637img/637lbl | test=32img/32lbl | valid=52img/52lbl
   [INFRARROJA] train=637img/637lbl | test=32img/32lbl | valid=52img/52lbl
   [NOCTURNO] train=637img/637lbl | test=32img/32lbl | valid=52img/52lbl

📁 OVEJAS -> clase 1 (oveja)
   [NORMAL] train=2378img/597lbl | test=304img/85lbl | valid=933img/238lbl
   [INFRARROJA] train=2378img/597lbl | test=304img/85lbl | valid=933img/238lbl
   [NOCTURNO] train=2378img/597lbl | test=304img/85lbl | valid=933img/238lbl

📁 GALLINAS -> clase 2 (gallina)
   [NORMAL] train=587img/587lbl | test=30img/30lbl | valid=50img/50lbl
   [INFRARROJA] train=587img/587lbl | test=30img/30lbl | valid=50img/50lbl
   [NOCTURNO] train=587img/587lbl | test=30img/30lbl | valid=50img/50lbl

📁 CUYES -> clase 3 (cuy)
   [NORMAL] train=2787img/2787lbl | test=143img/143lbl | valid=255img/255lbl
   [INFRARROJA] train=2787img/2787lbl | test=143img/143lbl | valid=255img/255lbl
   [NOCTURNO] train=2

---
## Celda 3 — Funciones de utilidad

In [4]:
def polygon_to_bbox(coords):
    xs = coords[0::2]
    ys = coords[1::2]
    if not xs or not ys:
        return None
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    cx = (x_min + x_max) / 2
    cy = (y_min + y_max) / 2
    w = x_max - x_min
    h = y_max - y_min
    return [cx, cy, w, h]

def convert_line_to_bbox(line):
    parts = list(map(float, line.strip().split()))
    if len(parts) < 5:
        return None
    cls_id = int(parts[0])
    coords = parts[1:]
    if len(coords) == 4:
        return f"{cls_id} {coords[0]:.6f} {coords[1]:.6f} {coords[2]:.6f} {coords[3]:.6f}"
    bbox = polygon_to_bbox(coords)
    if bbox:
        return f"{cls_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}"
    return None

def remap_label_content(content, old_to_new):
    new_lines = []
    for line in content.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            continue
        old_cls = int(parts[0])
        new_cls = old_to_new.get(old_cls)
        if new_cls is None:
            continue
        parts[0] = str(new_cls)
        bbox_line = convert_line_to_bbox(" ".join(parts))
        if bbox_line:
            new_lines.append(bbox_line)
    return new_lines

print("\u2705 Funciones cargadas")
print("  - polygon_to_bbox(): polígono a bounding box")
print("  - convert_line_to_bbox(): formato YOLO estándar")
print("  - remap_label_content(): reasignación de IDs de clase")

✅ Funciones cargadas
  - polygon_to_bbox(): polígono a bounding box
  - convert_line_to_bbox(): formato YOLO estándar
  - remap_label_content(): reasignación de IDs de clase


---
## Celda 4 — Consolidación del dataset unificado

Combina los 5 animales + 3 visiones en un solo dataset con clases remapeadas.

In [5]:
print("=" * 70)
print("CONSOLIDACIÓN DEL DATASET UNIFICADO")
print("=" * 70)

# Crear estructura unificada
for split in SPLITS:
    (UNIFIED_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (UNIFIED_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

stats = {s: {"copied": 0, "labels": 0} for s in SPLITS}
class_counter = Counter()

for animal, new_cls_id in CLASS_MAP.items():
    print(f"\n\U0001f4e6 Procesando {animal} -> clase {new_cls_id} ({CLASS_NAMES[new_cls_id]})")
    for vision in VISIONS:
        for split in SPLITS:
            src_images = list((BASE_DIR / animal / vision / split).glob("*.*"))
            valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
            src_images = [f for f in src_images if f.suffix.lower() in valid_exts]

            src_labels_dir = BASE_DIR / animal / vision / split / "labels"

            for img_path in tqdm(src_images, desc=f"  [{vision}/{split}]"):
                # Nombre único: animal_vision_nombreoriginal
                unique_name = f"{animal}_{vision}_{img_path.stem}{img_path.suffix}"
                dst_img = UNIFIED_DIR / split / "images" / unique_name
                shutil.copy2(img_path, dst_img)
                stats[split]["copied"] += 1

                # Label correspondiente
                lbl_path = src_labels_dir / f"{img_path.stem}.txt"
                if not lbl_path.exists() or lbl_path.stat().st_size == 0:
                    continue

                with open(lbl_path) as f:
                    original = f.read()

                new_lines = remap_label_content(original, {0: new_cls_id})
                if not new_lines:
                    continue

                dst_lbl = UNIFIED_DIR / split / "labels" / f"{Path(unique_name).stem}.txt"
                with open(dst_lbl, "w") as f:
                    f.write("\n".join(new_lines) + "\n")
                stats[split]["labels"] += 1
                class_counter[new_cls_id] += len(new_lines)

print("\n" + "=" * 70)
print("RESUMEN DE CONSOLIDACIÓN")
print("=" * 70)
for split, s in stats.items():
    print(f"  [{split}] {s['copied']} imágenes, {s['labels']} labels")

print("\nDistribución de clases:")
for cls_id in range(NUM_CLASSES):
    cnt = class_counter.get(cls_id, 0)
    print(f"  {cls_id}: {CLASS_NAMES[cls_id]} -> {cnt} anotaciones")

print(f"\n\U0001f4c1 Dataset unificado en: {UNIFIED_DIR}")

CONSOLIDACIÓN DEL DATASET UNIFICADO

📦 Procesando ZORROS -> clase 0 (zorro)


  [NORMAL/train]:   0%|          | 0/637 [00:00<?, ?it/s]

  [NORMAL/test]:   0%|          | 0/32 [00:00<?, ?it/s]

  [NORMAL/valid]:   0%|          | 0/52 [00:00<?, ?it/s]

  [INFRARROJA/train]:   0%|          | 0/637 [00:00<?, ?it/s]

...

  [INFRARROJA/valid]:   0%|          | 0/39 [00:00<?, ?it/s]

  [NOCTURNO/train]:   0%|          | 0/275 [00:00<?, ?it/s]

  [NOCTURNO/test]:   0%|          | 0/19 [00:00<?, ?it/s]

  [NOCTURNO/valid]:   0%|          | 0/39 [00:00<?, ?it/s]


RESUMEN DE CONSOLIDACIÓN
  [train] 19992 imágenes, 14643 labels
  [test] 1584 imágenes, 927 labels
  [valid] 3987 imágenes, 1902 labels

Distribución de clases:
  0: zorro -> 2268 anotaciones
  1: oveja -> 7290 anotaciones
  2: gallina -> 4890 anotaciones
  3: cuy -> 29790 anotaciones
  4: conejo -> 2088 anotaciones

📁 Dataset unificado en: d:\SOLEDAD VERSION FINAL\dataset_unificado


---
## Celda 5 — Generación del data.yaml

In [6]:
data_yaml = {
    "train": str(UNIFIED_DIR / "train"),
    "val": str(UNIFIED_DIR / "valid"),
    "test": str(UNIFIED_DIR / "test"),
    "nc": NUM_CLASSES,
    "names": CLASS_NAMES,
}

yaml_path = UNIFIED_DIR / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"\u2705 data.yaml generado: {yaml_path}")
print(f"\nContenido:")
with open(yaml_path) as f:
    print(f.read())

✅ data.yaml generado: d:\SOLEDAD VERSION FINAL\dataset_unificado\data.yaml

Contenido:
train: d:\SOLEDAD VERSION FINAL\dataset_unificado\train
val: d:\SOLEDAD VERSION FINAL\dataset_unificado\valid
test: d:\SOLEDAD VERSION FINAL\dataset_unificado\test
nc: 5
names:
- zorro
- oveja
- gallina
- cuy
- conejo



---
## Celda 6 — Entrenamiento del modelo

In [ ]:
print("=" * 70)
print(f"ENTRENAMIENTO: {CONFIG['model_name']}")
print(f"Clases: {CLASS_NAMES}")
print(f"Épocas: {CONFIG['epochs']} | Batch: {CONFIG['batch_size']} | Imgsz: {CONFIG['imgsz']}")
print(f"Dispositivo: {CONFIG['device']}")
print("=" * 70)

model = YOLO(CONFIG["model_name"])
print(f"\n\u2705 Modelo cargado: {CONFIG['model_name']}")

results = model.train(
    data=str(yaml_path),
    epochs=CONFIG["epochs"],
    batch=CONFIG["batch_size"],
    imgsz=CONFIG["imgsz"],
    patience=CONFIG["patience"],
    device=CONFIG["device"],
    workers=CONFIG["workers"],
    optimizer=CONFIG["optimizer"],
    lr0=CONFIG["lr0"],
    lrf=CONFIG["lrf"],
    momentum=CONFIG["momentum"],
    weight_decay=CONFIG["weight_decay"],
    warmup_epochs=CONFIG["warmup_epochs"],
    cos_lr=CONFIG["cos_lr"],
    close_mosaic=CONFIG["close_mosaic"],
    augment=CONFIG["augment"],
    seed=CONFIG["seed"],
    project=CONFIG["project"],
    name=CONFIG["name"],
    exist_ok=True,
    verbose=True,
)

print("\n" + "=" * 70)
print("\u2705 ENTRENAMIENTO COMPLETADO")
print("=" * 70)

ENTRENAMIENTO: yolo11n.pt
Clases: ['zorro', 'oveja', 'gallina', 'cuy', 'conejo']
Épocas: 100 | Batch: 8 | Imgsz: 640
Dispositivo: cuda:0

✅ Modelo cargado: yolo11n.pt
Ultralytics 8.4.60  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=d:\SOLEDAD VERSION FINAL\dataset_unificado\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4

: 

In [15]:
from pathlib import Path
import shutil, json, pandas as pd
from ultralytics import YOLO

runs_dir = Path("D:/SOLEDAD VERSION FINAL/output/runs")
print("Carpetas disponibles:")
for d in sorted(runs_dir.glob("pastor_guardian_*")):
    print(f"  {d.name}")

Carpetas disponibles:
  pastor_guardian_20260616_0203
  pastor_guardian_20260616_1107
  pastor_guardian_20260616_1118
  pastor_guardian_20260616_1123
  pastor_guardian_20260616_1126
  pastor_guardian_20260616_2232


In [16]:
from pathlib import Path
import shutil
from ultralytics import YOLO

best_pt = Path("D:/SOLEDAD VERSION FINAL/output/runs/pastor_guardian_20260616_2232/weights/best.pt")
final_dir = Path("D:/SOLEDAD VERSION FINAL/output/unificado")
final_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(best_pt, final_dir / "best.pt")
print(f"✅ Modelo listo: {final_dir / 'best.pt'}")

✅ Modelo listo: D:\SOLEDAD VERSION FINAL\output\unificado\best.pt


In [12]:
from pathlib import Path
import shutil, json, pandas as pd
from ultralytics import YOLO

exp_dir = Path("D:/SOLEDAD VERSION FINAL/output/runs/pastor_guardian_20260616_2232")
best_pt = exp_dir / "weights" / "best.pt"

if best_pt.exists():
    print(f"✅ Modelo recuperado: {best_pt}")
    print(f"   Tamaño: {best_pt.stat().st_size / 1024 / 1024:.1f} MB")
    
    # Validación rápida
    model = YOLO(str(best_pt))
    val = model.val(
        data="D:/SOLEDAD VERSION FINAL/dataset_unificado/data.yaml",
        split="test",
        batch=8,
        device=0,
        verbose=False
    )
    print(f"\n📊 mAP50: {val.box.map50:.3f} | Precision: {val.box.mp:.3f} | Recall: {val.box.mr:.3f}")
    
    # Copiar a output/unificado/
    final_dir = Path("D:/SOLEDAD VERSION FINAL/output/unificado")
    final_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(best_pt, final_dir / "best.pt")
    print(f"✅ Copiado a: {final_dir / 'best.pt'}")
else:
    print("❌ No se encontró best.pt en esa carpeta")

✅ Modelo recuperado: D:\SOLEDAD VERSION FINAL\output\runs\pastor_guardian_20260616_2232\weights\best.pt
   Tamaño: 5.2 MB
Ultralytics 8.4.60  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 146.980.5 MB/s, size: 95.4 KB)
val: Scanning D:\SOLEDAD VERSION FINAL\dataset_unificado\test\labels.cache... 927 images, 657 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1584/1584  0.0s


KeyboardInterrupt: 

---
## Celda 7 — Validación y métricas

In [ ]:
#ESTO TODAVIA NO CORRER

runs_dir = Path(CONFIG["project"])
exp_dirs = sorted(runs_dir.glob(f"{CONFIG['name']}*"))

if not exp_dirs:
    print("\u274c No hay entrenamiento. Ejecuta Celda 6.")
else:
    best = exp_dirs[-1] / "weights" / "best.pt"
    last = exp_dirs[-1] / "weights" / "last.pt"
    model_path = best if best.exists() else last

    if not model_path.exists():
        print(f"\u274c Modelo no encontrado en {exp_dirs[-1]}")
    else:
        print(f"\U0001f4e6 Validando: {model_path}")
        val_model = YOLO(str(model_path))
        val_results = val_model.val(
            data=str(yaml_path),
            split="test",
            batch=CONFIG["batch_size"],
            imgsz=CONFIG["imgsz"],
            conf=CONFIG["conf_thres"],
            iou=CONFIG["iou_thres"],
            device=CONFIG["device"],
            verbose=True,
        )

        print("\n" + "=" * 70)
        print("MÉTRICAS GLOBALES")
        print("=" * 70)
        global_metrics = {
            "Precision": val_results.box.mp,
	    "Recall": val_results.box.mr,
            "mAP@0.5": val_results.box.map50,
            "mAP@0.5:0.95": val_results.box.map,
        }
        df = pd.DataFrame([global_metrics])
        print(df.to_string(index=False))

        print("\nMÉTRICAS POR CLASE:")
        per_class = []
        for i, name in enumerate(CLASS_NAMES):
            per_class.append({
                "clase": f"{i} ({name})",
                "precision": val_results.box.p[i],
                "recall": val_results.box.r[i],
                "mAP50": val_results.box.ap50[i],
            })
        df2 = pd.DataFrame(per_class)
        print(df2.to_string(index=False))

        # Guardar
        metrics_path = exp_dirs[-1] / "metrics.json"
        with open(metrics_path, "w") as f:
            json.dump({"global": global_metrics, "per_class": per_class}, f, indent=2)
        print(f"\n\U0001f4c4 Métricas guardadas: {metrics_path}")

        # Copiar modelo final
        final_dir = OUTPUT_DIR / "unificado"
        final_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy(best, final_dir / "best.pt")
        print(f"\U0001f4e6 Modelo copiado: {final_dir / 'best.pt'}")

❌ No hay entrenamiento. Ejecuta Celda 6.


In [2]:
from ultralytics import YOLO
model = YOLO('D:/SOLEDAD VERSION FINAL/output/unificado/best.pt')
val = model.val(data='D:/SOLEDAD VERSION FINAL/dataset_unificado/data.yaml', split='test', batch=8, device=0, verbose=True)
print(f'\nPrecision: {val.box.mp:.3f}')
print(f'Recall: {val.box.mr:.3f}')
print(f'mAP50: {val.box.map50:.3f}')
print(f'mAP50-95: {val.box.map:.3f}')
for i, name in enumerate(['zorro','oveja','gallina','cuy','conejo']):
    print(f'  {name}: mAP50={val.box.ap50[i]:.3f}')

Ultralytics 8.4.60  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 17.811.2 MB/s, size: 117.3 KB)
val: Scanning D:\SOLEDAD VERSION FINAL\dataset_unificado\test\labels.cache... 927 images, 657 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1584/1584  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 198/198 19.3it/s 10.3s0.1s
                   all       1584       2538       0.85      0.694      0.774      0.548
                 zorro         96        102      0.946      0.882      0.966      0.679
                 oveja        255        750      0.807      0.697      0.746      0.493
               gallina         90        267      0.715      0.301      0.363      0.245
                   cuy        429       1314      0.915      0.847      0.929      0.688


In [3]:
from ultralytics import YOLO
model = YOLO('D:/SOLEDAD VERSION FINAL/output/unificado/best.pt')
print(model.model)  # muestra todas las capas

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
   

---
## Celda 8 — Visualización de resultados

In [ ]:
runs_dir = Path(CONFIG["project"])
exp_dirs = sorted(runs_dir.glob(f"{CONFIG['name']}*"))

if exp_dirs:
    results_csv = exp_dirs[-1] / "results.csv"
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = [c.strip() for c in df.columns]

        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        cols = {
            "train/box_loss": "Pérdida BBox",
            "train/cls_loss": "Pérdida Clasificación",
            "train/dfl_loss": "Pérdida DFL",
            "metrics/precision(B)": "Precisión",
            "metrics/recall(B)": "Recall",
            "metrics/mAP50(B)": "mAP@0.5",
        }
        for idx, (col, title) in enumerate(cols.items()):
            ax = axes[idx // 3][idx % 3]
            if col in df.columns:
                ax.plot(df["epoch"], df[col])
                ax.set_title(title)
                ax.set_xlabel("Época")
                ax.grid(True)

        plt.tight_layout()
        plt.show()

    # Mostrar predicciones de ejemplo
    final_model_path = OUTPUT_DIR / "unificado" / "best.pt"
    if final_model_path.exists():
        demo_model = YOLO(str(final_model_path))
        test_imgs = list((UNIFIED_DIR / "test" / "images").glob("*.*"))
        if test_imgs:
            sample = random.sample(test_imgs, min(6, len(test_imgs)))
            fig, axes = plt.subplots(2, 3, figsize=(18, 12))
            for ax, img_path in zip(axes.flatten(), sample):
                res = demo_model(str(img_path), conf=CONFIG["conf_thres"])
                frame = res[0].plot()
                ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                ax.axis("off")
                ax.set_title(img_path.stem[:30], fontsize=9)
            plt.tight_layout()
            plt.show()

---
## Celda 9 — Exportación del modelo final

In [17]:
final_model_path = OUTPUT_DIR / "unificado" / "best.pt"

if final_model_path.exists():
    export_model = YOLO(str(final_model_path))

    print("Exportando modelo...")
    onnx_path = export_model.export(format="onnx", imgsz=CONFIG["imgsz"])
    print(f"  \u2705 ONNX: {onnx_path}")

    # Copiar al directorio final
    shutil.copy(onnx_path, OUTPUT_DIR / "unificado" / "best.onnx")

    print(f"\n\U0001f4c1 Archivos en {OUTPUT_DIR / 'unificado'}:")
    for f in (OUTPUT_DIR / "unificado").iterdir():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  \U0001f4c4 {f.name} ({size_mb:.1f} MB)")
else:
    print("\u274c No hay modelo. Ejecuta Celda 6 primero.")

Exportando modelo...
Ultralytics 8.4.60  Python-3.12.10 torch-2.5.1+cu121 CPU (13th Gen Intel Core i9-13900H)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from 'd:\SOLEDAD VERSION FINAL\output\unificado\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
    --------------------------------------- 0.3/17.2 MB ? eta -:--:--
   - -------------------------------------- 0.8/17.2 MB 3.0 MB/s eta 0:00:06
   --- ------------------------------------ 1.3/17.2 MB 3.2 MB/s eta 0:00:05
   --- ------------------------------------ 1.6/17.2 MB 2.4 MB/s eta 0:00:07
   -

SameFileError: 'd:\\SOLEDAD VERSION FINAL\\output\\unificado\\best.onnx' and WindowsPath('d:/SOLEDAD VERSION FINAL/output/unificado/best.onnx') are the same file

In [18]:
pip install onnx onnxruntime onnxslim -q
Después ejecutá la exportación:
from ultralytics import YOLO
model = YOLO("D:/SOLEDAD VERSION FINAL/output/unificado/best.pt")

SyntaxError: invalid syntax (3884179894.py, line 1)

---
## Resumen Final

### Modelo generado
- `output/unificado/best.pt` — Pesos nativos PyTorch
- `output/unificado/best.onnx` — Exportado a ONNX

### Clases detectadas
| ID | Clase | Alerta |
|----|-------|--------|
| 0 | `zorro` | \u2757 Sí — activa alarma |
| 1 | `oveja` | \u2705 No |
| 2 | `gallina` | \u2705 No |
| 3 | `cuy` | \u2705 No |
| 4 | `conejo` | \u2705 No |

### Entrenamiento multi-visión
- \u2600\ufe0f **Normal**: imágenes originales (día)
- \U0001f4f7 **Infrarroja**: simulación térmica
- \U0001f303 **Nocturno**: simulación visión nocturna

El modelo funciona en cualquier condición de iluminación sin necesidad de cambiar de modelo.

### Para usar en producción:
```bash
python app.py
```
Abrir http://localhost:8000